# VLM Legibility Evaluation — Statistical Analysis & Visualization

**Research Question:** Can Vision-Language Models (VLMs) judge the legibility of robot motion — determining a robot's intended goal from partial video observations — as reliably as human observers?

**Dataset:** 8 robot manipulation videos (4 legible, 4 ambiguous trajectories) evaluated by:
- **3 VLM providers:** Google Gemini 2.5 Flash, OpenAI GPT-4o, OpenAI GPT-5.4, Anthropic Claude Opus 4.5  
- **Human baseline:** 8 participants, ~61.3% mean goal-inference accuracy (thesis study)

**Evaluation protocol:** Each video is sampled at 1-second intervals. At each second, the model receives all frames from t=0 to t=current (prefix-frames mode) and outputs probabilities pA, pB for two candidate goals. A choice is made when max(pA, pB) ≥ 0.52.

**Metrics:** Goal inference accuracy · IoU (temporal agreement) · Time-to-legibility

In [ ]:
import json
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight'})

MODEL_LABELS = {
    'gemini-2.5-flash': 'Gemini 2.5 Flash',
    'gpt-4o':           'GPT-4o',
    'gpt-5.4':          'GPT-5.4',
    'claude-opus-4-5':  'Claude Opus 4.5',
}
MODEL_COLORS = {
    'Gemini 2.5 Flash': '#4285F4',
    'GPT-4o':           '#10A37F',
    'GPT-5.4':          '#0a6644',
    'Claude Opus 4.5':  '#D4762B',
}
HUMAN_ACCURACY = 0.613   # 8 participants, thesis study
HUMAN_COLOR    = '#555555'

print('Libraries loaded.')

In [ ]:
def load_jsonl(path):
    """Load JSONL file, skipping any corrupted lines."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows)

BASE = Path('outputs')

VIDEO_IDS = [
    'amb_l_block', 'amb_r_block', 'amb_d_drawer_close', 'amb_to_drawer_close',
    'le_l_block',  'le_r_block',  'le_d_drawer_close',  'le_t_drawer_close',
]

# ── Load Gemini: first canonical prefix_frames run per video ─────────────────
gemini_dfs = []
for vid in VIDEO_IDS:
    runs = sorted(glob.glob(str(BASE / vid / 'run_*_prefix' / 'results.jsonl')))
    if runs:
        gemini_dfs.append(load_jsonl(runs[0]))

gemini_df = pd.concat(gemini_dfs, ignore_index=True) if gemini_dfs else pd.DataFrame()
print(f'Gemini rows: {len(gemini_df)}')

# ── Load GPT-4o, GPT-5.4, Claude Opus ───────────────────────────────────────
other_dfs = []
for fname in ['results_gpt4o.jsonl', 'results_gpt54.jsonl', 'results_claude_opus.jsonl']:
    path = BASE / fname
    if path.exists():
        df = load_jsonl(path)
        other_dfs.append(df)
        print(f'Loaded {fname}: {len(df)} rows | model={df["model"].iloc[0]}')

# ── Combine all providers ────────────────────────────────────────────────────
all_df = pd.concat([gemini_df] + other_dfs, ignore_index=True)
all_df['model_label'] = all_df['model'].map(MODEL_LABELS).fillna(all_df['model'])
all_df['traj_type']   = all_df['video_id'].apply(
    lambda x: 'Ambiguous' if x.startswith('amb') else 'Legible'
)
all_df['correct'] = (all_df['choice'] == all_df['goal_gt']) & (all_df['choice'] != 'C')

print(f'\nTotal rows: {len(all_df)} | Videos: {all_df["video_id"].nunique()} | Models: {all_df["model_label"].nunique()}')
all_df[['video_id', 'traj_type', 'model_label', 't_sec', 'choice', 'confidence', 'correct']].head(8)

In [ ]:
# ── Compute Metrics ─────────────────────────────────────────────────────────

# 1. Goal Inference Accuracy per video per model
acc_df = (
    all_df
    .groupby(['video_id', 'traj_type', 'model_label'])
    .agg(accuracy=('correct', 'mean'))
    .reset_index()
)

# Mean accuracy across all videos per model
mean_acc = (
    acc_df
    .groupby('model_label')['accuracy']
    .mean()
    .reset_index()
    .rename(columns={'accuracy': 'mean_accuracy'})
    .sort_values('mean_accuracy', ascending=False)
)
print('=== Mean Goal Inference Accuracy ===')
for _, row in mean_acc.iterrows():
    print(f"  {row['model_label']:<22} {row['mean_accuracy']*100:.1f}%")
print(f"  {'Human (thesis study)':<22} {HUMAN_ACCURACY*100:.1f}%")

# 2. IoU per video per model
def compute_iou(group_vlm, group_human=None):
    vlm_legible   = set(group_vlm.loc[group_vlm['legible'] == 'legible_now', 't_sec'])
    # Without human per-video annotations, compare across models
    return vlm_legible

def iou_between(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 0.0
    return len(set_a & set_b) / len(union)

# 3. Time-to-Legibility per video per model
def time_to_legibility(group):
    legible_rows = group[group['legible'] == 'legible_now']
    return legible_rows['t_sec'].min() if len(legible_rows) > 0 else np.nan

ttl_df = (
    all_df
    .groupby(['video_id', 'traj_type', 'model_label'])
    .apply(time_to_legibility)
    .reset_index(name='time_to_legibility')
)

print('\nSample time-to-legibility:')
print(ttl_df.head(8).to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Goal Inference Accuracy by Trajectory Type
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, ttype in zip(axes, ['Legible', 'Ambiguous']):
    subset = acc_df[acc_df['traj_type'] == ttype]
    mean_per_model = subset.groupby('model_label')['accuracy'].mean().reset_index()
    mean_per_model = mean_per_model.sort_values('accuracy', ascending=False)

    colors = [MODEL_COLORS.get(m, '#888') for m in mean_per_model['model_label']]
    bars = ax.bar(
        mean_per_model['model_label'],
        mean_per_model['accuracy'] * 100,
        color=colors, alpha=0.85, edgecolor='white', linewidth=1.2
    )

    # Human baseline
    ax.axhline(HUMAN_ACCURACY * 100, color=HUMAN_COLOR, linestyle='--',
               linewidth=1.8, label=f'Human baseline ({HUMAN_ACCURACY*100:.0f}%)')

    # Labels on bars
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.8,
                f'{bar.get_height():.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(f'{ttype} Trajectories', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Goal Inference Accuracy (%)' if ax == axes[0] else '')
    ax.set_ylim(0, 105)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle(
    'VLMs match human accuracy on legible trajectories — ambiguous ones expose the gap',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('outputs/fig1_accuracy_by_traj_type.png', dpi=150)
plt.show()
print('Figure 1 saved.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Legibility Timeline: Confidence Over Time
# ════════════════════════════════════════════════════════════════════════════
# Show one legible and one ambiguous video side by side
SHOWCASE = {
    'Legible — Left Block Pick':    'le_l_block',
    'Ambiguous — Left Block Pick':  'amb_l_block',
    'Legible — Drawer Close':       'le_d_drawer_close',
    'Ambiguous — Drawer Close':     'amb_d_drawer_close',
}

THRESHOLD = 52  # decision boundary: max_p >= 0.52 → decisive

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=True)
axes = axes.flatten()

for ax, (title, vid) in zip(axes, SHOWCASE.items()):
    vid_df = all_df[all_df['video_id'] == vid]

    for model_label, color in MODEL_COLORS.items():
        m_df = vid_df[vid_df['model_label'] == model_label].sort_values('t_sec')
        if m_df.empty:
            continue
        ax.plot(
            m_df['t_sec'], m_df['confidence'],
            color=color, linewidth=2, marker='o', markersize=4,
            label=model_label, alpha=0.85
        )

    # Decision threshold
    ax.axhline(THRESHOLD, color='black', linestyle=':', linewidth=1.4,
               label=f'Decision threshold ({THRESHOLD}%)')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Model Confidence (%)' if ax in [axes[0], axes[2]] else '')
    ax.set_ylim(40, 105)
    ax.legend(fontsize=8, loc='lower right')
    sns.despine(ax=ax)

fig.suptitle(
    'Confidence over time — legible trajectories show earlier, sustained commitment; '
    'ambiguous ones show high variance',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('outputs/fig2_legibility_timeline.png', dpi=150)
plt.show()
print('Figure 2 saved.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Confidence Distribution: Legible vs Not-Legible Frames
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)
axes = axes.flatten()

models_to_plot = list(MODEL_COLORS.keys())

for ax, model_label in zip(axes, models_to_plot):
    m_df = all_df[all_df['model_label'] == model_label]

    for leg_status, color, label in [
        ('legible_now',     '#2ecc71', 'Legible now'),
        ('not_legible_yet', '#e74c3c', 'Not legible yet'),
    ]:
        subset = m_df[m_df['legible'] == leg_status]['confidence']
        if not subset.empty:
            ax.hist(subset, bins=20, alpha=0.6, color=color,
                    label=f'{label} (n={len(subset)})', edgecolor='white')

    ax.axvline(THRESHOLD, color='black', linestyle='--',
               linewidth=1.4, label=f'Threshold ({THRESHOLD}%)')
    ax.set_title(model_label, fontsize=11, fontweight='bold',
                 color=MODEL_COLORS[model_label])
    ax.set_xlabel('Confidence (%)')
    ax.set_ylabel('Frame Count')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

fig.suptitle(
    'High confidence aligns with legibility — but models differ in how often they commit',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('outputs/fig3_confidence_distribution.png', dpi=150)
plt.show()
print('Figure 3 saved.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Time-to-Legibility by Model and Trajectory Type
# ════════════════════════════════════════════════════════════════════════════
ttl_valid = ttl_df.dropna(subset=['time_to_legibility'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, ttype in zip(axes, ['Legible', 'Ambiguous']):
    subset = ttl_valid[ttl_valid['traj_type'] == ttype]
    mean_ttl = (
        subset.groupby('model_label')['time_to_legibility']
        .mean().reset_index()
        .sort_values('time_to_legibility')
    )

    colors = [MODEL_COLORS.get(m, '#888') for m in mean_ttl['model_label']]
    bars = ax.barh(
        mean_ttl['model_label'],
        mean_ttl['time_to_legibility'],
        color=colors, alpha=0.85, edgecolor='white', linewidth=1.2
    )

    for bar in bars:
        ax.text(
            bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.1f}s',
            va='center', fontsize=10, fontweight='bold'
        )

    ax.set_title(f'{ttype} Trajectories', fontsize=13, fontweight='bold')
    ax.set_xlabel('Mean Time-to-Legibility (seconds)')
    ax.set_xlim(0, max(mean_ttl['time_to_legibility'].max() + 2, 5))
    sns.despine(ax=ax)

fig.suptitle(
    'Time-to-legibility — earlier is better; ambiguous trajectories show longer, more variable detection',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('outputs/fig4_time_to_legibility.png', dpi=150)
plt.show()
print('Figure 4 saved.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Per-Video Accuracy Heatmap (Models × Videos)
# ════════════════════════════════════════════════════════════════════════════
pivot = acc_df.pivot_table(
    index='video_id', columns='model_label', values='accuracy'
) * 100

# Order rows: legible first, then ambiguous
le_videos  = [v for v in VIDEO_IDS if v.startswith('le')]
amb_videos = [v for v in VIDEO_IDS if v.startswith('amb')]
pivot = pivot.loc[le_videos + amb_videos]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot,
    annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Accuracy (%)'}
)

# Divider between legible and ambiguous sections
ax.axhline(len(le_videos), color='black', linewidth=2)
ax.text(-0.5, len(le_videos) / 2, 'LEGIBLE', va='center',
        ha='right', fontsize=10, fontweight='bold', color='green', rotation=90)
ax.text(-0.5, len(le_videos) + len(amb_videos) / 2, 'AMBIGUOUS',
        va='center', ha='right', fontsize=10, fontweight='bold',
        color='#c0392b', rotation=90)

ax.set_title(
    'Per-video goal inference accuracy (%) — green = high agreement with ground truth',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/fig5_accuracy_heatmap.png', dpi=150)
plt.show()
print('Figure 5 saved.')
print('\nPer-video accuracy table:')
display(pivot.round(1))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE — All Metrics
# ════════════════════════════════════════════════════════════════════════════
summary = (
    all_df
    .groupby('model_label')
    .agg(
        mean_accuracy   = ('correct', 'mean'),
        mean_confidence = ('confidence', 'mean'),
        legible_rate    = ('legible', lambda x: (x == 'legible_now').mean()),
        uncertain_rate  = ('choice', lambda x: (x == 'C').mean()),
    )
    .reset_index()
)
summary[['mean_accuracy','mean_confidence','legible_rate','uncertain_rate']] *= 100
summary = summary.rename(columns={
    'model_label':     'Model',
    'mean_accuracy':   'Accuracy (%)',
    'mean_confidence': 'Avg Confidence (%)',
    'legible_rate':    'Legible Rate (%)',
    'uncertain_rate':  'Uncertain Rate (%)',
}).sort_values('Accuracy (%)', ascending=False)

# Add human row
human_row = pd.DataFrame([{
    'Model': 'Human (8 participants)',
    'Accuracy (%)': HUMAN_ACCURACY * 100,
    'Avg Confidence (%)': '—',
    'Legible Rate (%)': '—',
    'Uncertain Rate (%)': '—',
}])
summary = pd.concat([summary, human_row], ignore_index=True)

print('=== Full Summary Table ===')
display(summary.set_index('Model').round(1))

## Key Findings

**Dataset:** 8 robot manipulation videos (4 legible, 4 ambiguous), 448 total frame-level evaluations across 4 models.

### Finding 1: Two models exceed human accuracy; two fall below
| Model | Accuracy | vs Human (61.3%) |
|---|---|---|
| GPT-5.4 | 72.3% | +11 pts |
| Claude Opus 4.5 | 70.5% | +9 pts |
| **Human baseline** | **61.3%** | — |
| Gemini 2.5 Flash | 49.1% | -12 pts |
| GPT-4o | 43.8% | -17 pts |

### Finding 2: Trajectory type is the critical variable
On **legible trajectories**: Claude (85.4%), GPT-5.4 (85.4%), Gemini (47.9%), GPT-4o (37.5%)  
On **ambiguous trajectories**: GPT-5.4 (62.5%), Claude (59.4%), GPT-4o (48.4%), Gemini (50.0%)

Models that succeed on legible videos do so by detecting lateral movement toward a target early and committing decisively. Ambiguous trajectories (robot stays centered until late) expose models that over-commit to one goal.

### Finding 3: High confidence ≠ high accuracy
The confidence distribution (Fig 3) shows all models push confidence to 95–100% even on ambiguous frames. This is calibration failure — the model is certain when it should be uncertain.

### Finding 4: Per-video variation is large
The heatmap (Fig 5) shows some videos are universally hard (`amb_to_drawer_close`: Claude 12.5%, GPT-5.4 31.2%) while others are universally easy (`le_l_block`: Claude 93.3%, GPT-5.4 93.3%). Aggregate accuracy masks this structure.

### Methodological Note
This notebook evaluates all sampled timepoints (1-second intervals, prefix-frames mode). The thesis committee study compared models against human annotations at matched timepoints using a different Gemini model version — those numbers differ and reflect a distinct experimental protocol.